# CSE 253R Assignment 2 — Task 4: MusicGen Fine-tuning (Colab)

**Before you start:**
1. Go to **Runtime → Change runtime type → T4 GPU**
2. Push your latest code to GitHub (or upload this folder to Drive)
3. Set `SMOKE_TEST = True` first (~30 min), then `False` for the full run (~4–6 hrs)

**What this notebook does:**
- Downloads FMA-small from HuggingFace
- Fine-tunes `facebook/musicgen-small` on 4 genres
- Generates `continuous_conditioned.mp3` (main deliverable)
- Zips outputs for download to your Mac

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────
SMOKE_TEST = True   # True = quick test (20 tracks, 5 epochs). False = full run.

# Your GitHub repo (update if different)
REPO_URL = "https://github.com/achirra-cypher/cse253r-assignment2.git"
BRANCH = "main"

# Optional: mount Google Drive to save checkpoints if Colab disconnects
USE_DRIVE = False   # set True to copy finetuned checkpoint to Drive
DRIVE_FOLDER = "cse253r-assignment2"

## Step 0 — Check GPU

In [ ]:
!nvidia-smi

import torch
assert torch.cuda.is_available(), (
    "No GPU detected! Go to Runtime → Change runtime type → T4 GPU, then re-run this cell."
)
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Step 1 — Clone repo & install dependencies

In [ ]:
import os
from pathlib import Path

PROJECT_DIR = Path("/content/cse253r-assignment2")

if PROJECT_DIR.exists():
    !rm -rf {PROJECT_DIR}

!git clone -b {BRANCH} {REPO_URL} {PROJECT_DIR}
%cd {PROJECT_DIR}

!pip install -q datasets soundfile librosa transformers accelerate torchaudio scipy scikit-learn

print("\nProject files:")
for f in ["prepare_fma_data.py", "musicgen_finetune.py", "musicgen_generate.py"]:
    print(f"  {'✓' if Path(f).exists() else '✗'} {f}")

## Step 2 — (Optional) Mount Google Drive for backup

In [ ]:
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    drive_path = Path(f"/content/drive/MyDrive/{DRIVE_FOLDER}")
    drive_path.mkdir(parents=True, exist_ok=True)
    print(f"Drive backup folder: {drive_path}")
else:
    print("Skipping Drive mount (USE_DRIVE=False)")

## Step 3 — Download & prepare FMA data

Downloads from HuggingFace `rpmon/fma-genre-classification` and writes
`musicgen_data/train/` and `musicgen_data/valid/` with WAV + JSON pairs.

**Full run:** 4 genres × 200 tracks ≈ 30–45 min download + processing.

In [ ]:
N_SAMPLES = 20 if SMOKE_TEST else 200
print(f"Mode: {'SMOKE TEST' if SMOKE_TEST else 'FULL RUN'} — {N_SAMPLES} tracks per genre\n")

!python prepare_fma_data.py --n-samples {N_SAMPLES}

In [ ]:
# Verify data was created
from pathlib import Path
import json

info_path = Path("musicgen_data/dataset_info.json")
if info_path.exists():
    with open(info_path) as f:
        info = json.load(f)
    print("Dataset info:", json.dumps(info, indent=2))
else:
    raise FileNotFoundError("prepare_fma_data.py did not create musicgen_data/")

wav_count = len(list(Path("musicgen_data").rglob("*.wav")))
print(f"\nTotal WAV files: {wav_count}")

## Step 4 — Fine-tune MusicGen-small

Fine-tunes the transformer decoder on (audio, text description) pairs.

| Mode | Epochs | Approx time (T4) |
|------|--------|------------------|
| Smoke test | 5 | ~20–30 min |
| Full run | 25 | ~4–6 hours |

**Tip:** Colab free tier may disconnect. Set `USE_DRIVE=True` above to backup the checkpoint.

In [ ]:
EPOCHS = 5 if SMOKE_TEST else 25
BATCH_SIZE = 2   # keep at 2 for T4 (16 GB VRAM)

!python musicgen_finetune.py --epochs {EPOCHS} --batch-size {BATCH_SIZE}

In [ ]:
# Backup checkpoint to Drive (optional)
if USE_DRIVE:
    import shutil
    src = Path("finetuned_musicgen")
    dst = drive_path / "finetuned_musicgen"
    if src.exists():
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print(f"Checkpoint backed up to {dst}")

## Step 5 — Generate audio samples

Generates one sample per genre + the main deliverable `continuous_conditioned.mp3`.

In [ ]:
# Generate samples for all 4 genres (fine-tuned model)
!python musicgen_generate.py --checkpoint finetuned_musicgen --all-genres --out-dir generated_audio_finetuned

# Also generate pretrained baseline for comparison (Day 4 evaluation)
!python musicgen_generate.py --all-genres --out-dir generated_audio

# Main deliverable
!python musicgen_generate.py --checkpoint finetuned_musicgen --prompt "hip hop music with beats and rhythm" --output continuous_conditioned.mp3 --duration 30

In [ ]:
# Listen to generated samples in Colab
from IPython.display import Audio, display
from pathlib import Path

for folder in ["generated_audio_finetuned", "generated_audio"]:
    p = Path(folder)
    if not p.exists():
        continue
    print(f"\n=== {folder} ===")
    for f in sorted(p.glob("*")):
        if f.suffix in (".wav", ".mp3"):
            print(f.name)
            display(Audio(filename=str(f)))

if Path("continuous_conditioned.mp3").exists():
    print("\n★ Main deliverable: continuous_conditioned.mp3")
    display(Audio(filename="continuous_conditioned.mp3"))
elif Path("continuous_conditioned.wav").exists():
    print("\n★ Main deliverable: continuous_conditioned.wav (ffmpeg not available for mp3)")
    display(Audio(filename="continuous_conditioned.wav"))

## Step 6 — Task 4 evaluation (optional)

In [ ]:
!python evaluate_task4.py

## Step 7 — Download results to your Mac

Downloads a zip with everything you need for submission and local evaluation.

In [ ]:
import shutil
from pathlib import Path
from google.colab import files

zip_name = "task4_outputs"
staging = Path(f"/content/{zip_name}")
if staging.exists():
    shutil.rmtree(staging)
staging.mkdir()

# Copy key outputs
to_copy = [
    "continuous_conditioned.mp3",
    "continuous_conditioned.wav",
    "finetune_history.json",
    "evaluation_task4.json",
    "eval_task4_genre_accuracy.png",
]
for name in to_copy:
    src = Path(name)
    if src.exists():
        shutil.copy(src, staging / name)

for folder in ["generated_audio", "generated_audio_finetuned"]:
    src = Path(folder)
    if src.exists():
        shutil.copytree(src, staging / folder)

# Checkpoint is large (~1.2 GB) — copy separately if needed
ckpt = Path("finetuned_musicgen")
if ckpt.exists():
    print(f"finetuned_musicgen/ is {sum(f.stat().st_size for f in ckpt.rglob('*') if f.is_file()) / 1e9:.2f} GB")
    print("To include checkpoint in zip, uncomment the line below (large download!)")
    # shutil.copytree(ckpt, staging / "finetuned_musicgen")

shutil.make_archive(f"/content/{zip_name}", "zip", staging)
print(f"Created /content/{zip_name}.zip")
files.download(f"/content/{zip_name}.zip")

## After downloading — on your Mac

1. Unzip into your project folder
2. Confirm `continuous_conditioned.mp3` is present
3. Run locally:
   ```bash
   source .venv/bin/activate
   python evaluate_task4.py
   jupyter notebook workbook.ipynb   # run Section 4 cells
   ```
4. Continue with Day 5 (export `workbook.html`) and Day 6 (video)

---

### Troubleshooting

| Problem | Fix |
|---------|-----|
| `No GPU detected` | Runtime → Change runtime type → T4 GPU |
| Colab disconnected mid-training | Set `USE_DRIVE=True`, re-run from Step 4 (data prep is cached if you saved Drive copy) |
| `Out of memory` | Reduce `--batch-size` to 1 in Step 4 |
| HuggingFace download slow | Normal on first run; model caches afterward |
| `continuous_conditioned.wav` instead of `.mp3` | Install ffmpeg in Colab: `!apt-get install -y ffmpeg` then re-run Step 5 |
| Repo clone fails | Update `REPO_URL` in Step 0 config cell to your fork |